# Full-Scale GRU4Rec Training on Colab (GPU)

**Why this notebook exists:** this repo's training script (`src/model/train.py`) is CPU-friendly (sampled softmax, small vocab), but the GRU recurrence and embedding lookups still run meaningfully faster on a GPU's cuDNN-accelerated kernels. Colab's free-tier GPU (usually a T4) can turn the full 20M-row run from ~2.5-3 hours/epoch down to a few minutes/epoch.

**The one thing this needs that's different from local:** Colab can't reach your local MySQL database. This notebook never queries MySQL -- it trains on a Parquet file you export *locally* first and upload, same as the rest of the pipeline already does (MySQL -> Parquet is the same bridge `export.py` always used).

## One-time setup, done on your local machine first

1. Export the training window to Parquet (this already talks to MySQL, so it has to run locally, not in Colab):
   ```
   .venv\Scripts\python.exe -m src.data.export
   ```
   This writes `data/interim/events.parquet`. For the full run, don't pass `--start-date`/`--end-date` -- the default is the whole loaded range.
2. Zip the `src/` folder (it's small, just code -- no data in it).
3. Upload both `src.zip` and `data/interim/events.parquet` to a folder in your Google Drive, e.g. `MyDrive/product-recommender/`.
4. Open this notebook in Colab (upload it, or open from Drive), and set **Runtime -> Change runtime type -> GPU** before running anything below.

Everything from here on runs *in Colab*.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Adjust this to wherever you uploaded src.zip and events.parquet
DRIVE_PROJECT_DIR = "/content/drive/MyDrive/product-recommender"

In [ ]:
import shutil
import sys
from pathlib import Path

import tensorflow as tf

gpus = tf.config.list_physical_devices("GPU")
print(f"GPUs visible to TensorFlow: {gpus}")
if not gpus:
    print("WARNING: no GPU detected -- check Runtime > Change runtime type > GPU, then re-run this cell.")

WORK_DIR = Path("/content/product-recommender")
WORK_DIR.mkdir(exist_ok=True)

# Unzip the code into a local (fast) disk rather than importing straight off Drive.
shutil.unpack_archive(f"{DRIVE_PROJECT_DIR}/src.zip", WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print("Contents:", list(WORK_DIR.iterdir()))

## Install missing dependencies

Colab already ships TensorFlow, pandas, and pyarrow -- we only need what's
missing. `mysql-connector-python` is deliberately **not** installed here;
this notebook never imports `src.data.db`/`src.data.export`, so it's not
needed, and Colab couldn't reach your local MySQL instance anyway.

In [ ]:
!pip install -q python-dotenv scikit-learn

## Train

This calls the *exact same* `train()` function from `src/model/train.py` that
runs locally -- same model, same sampled softmax, same early stopping on
validation Recall@10. The only difference is `skip_export=True`: instead of
querying MySQL, it reads the Parquet file you uploaded. Everything else in
the pipeline (session building, vocab, time-based split, training,
evaluation) is identical to the local run -- that's the point of having it
as one shared, tested module instead of duplicated notebook code.

In [ ]:
from src.model.train import train

model = train(
    skip_export=True,
    events_parquet_path=Path(f"{DRIVE_PROJECT_DIR}/events.parquet"),
    # epochs/patience/test_days/val_days default to src/config.py's full-scale
    # settings (EPOCHS=10, EARLY_STOPPING_PATIENCE=2, TEST_DAYS=3, VAL_DAYS=3) --
    # override here only if this Parquet file covers a narrower window.
)

## Save results back to Drive

`train()` already wrote `gru4rec.weights.h5` and `item_vocab.json` under
`WORK_DIR/models/` (same relative path `src/config.py` uses locally). Copy
them back to Drive so they survive when this Colab runtime recycles.

In [ ]:
import shutil

output_dir = Path(f"{DRIVE_PROJECT_DIR}/trained_model")
output_dir.mkdir(exist_ok=True)

shutil.copy(WORK_DIR / "models" / "gru4rec.weights.h5", output_dir)
shutil.copy(WORK_DIR / "models" / "item_vocab.json", output_dir)

print(f"Saved to {output_dir} -- download both files and drop them into your local models/ folder.")